# Alpha Research — Rolling Window Backtest

### Objective
Implement a proper walk-forward rolling window backtest with no look-ahead bias.

- At each date T, train on trailing 252-day window (strictly before T)
- Features z-scored cross-sectionally across stocks at each date using only training data statistics
- Target: market-adjusted excess return (stock return minus SPY return)
- Universe: S&P 500 point-in-time constituents as of 2010-01-01

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import mstats
import yfinance as yf

DATA = '../../seminar_02/homework/data/'

# ── Load raw panel (unstandardized features from HW2) ─────────────────
panel = pd.read_csv(DATA + 'panel_clean.csv', parse_dates=['date'])

# ── Load prices for computing additional features ──────────────────────
prices_clean = pd.read_csv(DATA + 'sp500_prices_clean.csv', index_col=0, parse_dates=True)

# ── Load SPY for market-adjusted target ───────────────────────────────
spy = yf.download('SPY', start='2010-01-01', end='2026-05-16', auto_adjust=True)['Close'].squeeze()

print(f'Panel shape:  {panel.shape}')
print(f'Prices shape: {prices_clean.shape}')
print(f'Panel columns: {list(panel.columns)}')

In [ ]:
# ── Compute raw (unstandardized) features ────────────────────────────
# These will be z-scored cross-sectionally at each date in the rolling loop

# 3-month and 6-month momentum (raw log returns)
mom_3m = np.log(prices_clean).diff(63).shift(21)
mom_6m = np.log(prices_clean).diff(126).shift(21)

# Short-term reversal (raw 21-day return)
reversal = prices_clean.pct_change(21)

# Volatility (raw 21-day rolling std)
volatility = prices_clean.pct_change().rolling(21).std()

# 52-week high ratio (bounded 0-1, no standardization needed)
high_52w = prices_clean.rolling(252).max()
ratio_52w = prices_clean / high_52w

# Melt all to long format
def to_long(df, name):
    long = df.stack().reset_index()
    long.columns = ['date', 'ticker', name]
    return long

mom_3m_long   = to_long(mom_3m,    'mom_3m')
mom_6m_long   = to_long(mom_6m,    'mom_6m')
reversal_long = to_long(reversal,  'reversal')
vol_long      = to_long(volatility,'volatility')
ratio_long    = to_long(ratio_52w, 'ratio_52w')

# ── Build raw panel ───────────────────────────────────────────────────
panel_raw = panel[['date', 'ticker', 'momentum', 'forward_return']].copy()

for df in [mom_3m_long, mom_6m_long, reversal_long, vol_long, ratio_long]:
    panel_raw = panel_raw.merge(df, on=['date', 'ticker'], how='left')

# ── Market-adjusted target ────────────────────────────────────────────
spy_21d = spy.pct_change(21).shift(-21)
spy_21d.name = 'spy_21d'
panel_raw = panel_raw.merge(spy_21d.reset_index().rename(columns={'Date':'date'}), on='date', how='left')
panel_raw['excess_return'] = panel_raw['forward_return'] - panel_raw['spy_21d']

panel_raw = panel_raw.dropna()
panel_raw = panel_raw.sort_values('date')

print(f'Raw panel shape: {panel_raw.shape}')
print(f'Columns: {list(panel_raw.columns)}')